In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import sys
print("torch" in sys.modules)

Import the dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("CADCODER/GenCAD-Code")

train = ds['train']
test = ds['test']
eval = ds['validation']

In [ ]:
import random
random.seed(42)
eval_subset = eval.shuffle(seed=42).select(range(20))

Download the checkpoint

In [ ]:
from huggingface_hub import create_repo


HUB_REPO = ""
os.environ["HF_TOKEN"] = ""
checkpoint = ""

# create_repo(repo_id=HUB_REPO, private=True, exist_ok=True)



from gdrive_fsspec import GoogleDriveFileSystem

fs = GoogleDriveFileSystem(use_listings_cache=False, skip_instance_cache=True, 
                           auth_kwargs={"use_local_webserver": False})
fs.get("", "", recursive=True)


sft_path = ""

if checkpoint != "sft" :
    from huggingface_hub import hf_hub_download, snapshot_download


    path = snapshot_download(repo_id= HUB_REPO,allow_patterns=f"{checkpoint}/*", local_dir="RL_checkpoint")

print(path)

Load the model

In [ ]:
import accelerate
from transformers import AutoProcessor, AutoModelForMultimodalLM
from peft import PeftModel

processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-4B")
base_model = AutoModelForMultimodalLM.from_pretrained("Qwen/Qwen3.5-4B", device_map="cuda")

model = PeftModel.from_pretrained(base_model, sft_path, is_trainable = True)
# model = base_model
model.eval()

In [ ]:
model.print_trainable_parameters()

To extract only the python code and nothing else from what the model returns:

In [ ]:
import re

_CODE_FENCE_RE = re.compile(r'```(?:python)?\s*\n?(.*?)\n?```', re.DOTALL)

def extract_code(text):
    match = _CODE_FENCE_RE.search(text)
    if match:
        return match.group(1).strip()
    return text.strip()

In [ ]:
INSTRUCTION = "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words."

def to_example(row):
    return {
        "prompt": [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": INSTRUCTION}]}],
        "images": [row["image"]],
        "gt_code": row["cadquery"],
    }

train_ds = train.shuffle(seed = 42).select(range(5000))
train_ds_mapped = train_ds.map(to_example, num_proc = 4)
eval_subset_mapped = eval_subset.map(to_example, num_proc=4)

In [ ]:
import subprocess, tempfile
from pathlib import Path
import numpy as np

from best_iou import _load_solid_from_code, _normalized_mesh, iou_best

BEST_IOU_DIR = ""

def execute_and_get_mesh(code: str, timeout: int = 300):
    with tempfile.TemporaryDirectory() as td:
        mesh_out = Path(td) / "mesh.npz"
        script_text = f'''
import sys, numpy as np
sys.path.insert(0, {BEST_IOU_DIR!r})
from best_iou import _load_solid_from_code, _normalized_mesh, iou_best

code = {code!r}
solid = _load_solid_from_code(code)
mesh = _normalized_mesh(solid)
np.savez({str(mesh_out)!r}, vertices=mesh.vertices, faces=mesh.faces)
'''
        with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
            f.write(script_text)
            tmp = f.name
        try:
            r = subprocess.run([sys.executable, tmp], timeout=timeout, capture_output=True, text=True)
        except subprocess.TimeoutExpired as e:
            raise RuntimeError(f"timeout after {timeout}s") from e
        finally:
            os.unlink(tmp)

        if r.returncode != 0:
            raise RuntimeError(r.stderr.strip().splitlines()[-1] if r.stderr else "unknown error")
        if not mesh_out.exists():
            raise RuntimeError("no mesh file written")

        data = np.load(mesh_out)
        import trimesh
        return trimesh.Trimesh(vertices=data["vertices"], faces=data["faces"])

In [ ]:
def voxel_iou_reward(completions, gt_code, **kwargs):
    scores = []
    for completion, gt in zip(completions, gt_code):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        code = extract_code(text)
        if not code:
            scores.append(0.0)
            continue
        try:
            mesh_model = execute_and_get_mesh(code)
            mesh_gt = execute_and_get_mesh(gt)
            scores.append(iou_best(mesh_gt, mesh_model))
        except Exception:
            scores.append(0.0)
    return scores

In [ ]:
from huggingface_hub import HfApi
from transformers import TrainerCallback

api = HfApi()

class FullCheckpointPushCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        ckpt_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        try:
            api.upload_folder(
                folder_path=str(ckpt_dir),
                repo_id=HUB_REPO,
                path_in_repo=f"checkpoint-{state.global_step}",
                commit_message=f"checkpoint step {state.global_step}",
            )
        except Exception as e:
            print(f"checkpoint push failed at step {state.global_step}: {e}", flush=True)

In [ ]:
class PrintLogCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            print(f"step {state.global_step}: {logs}", flush=True)

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="/rl",  
    learning_rate=1e-6,
    remove_unused_columns=False,
    num_generations=8,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    max_completion_length=2560,
    gradient_checkpointing=True,
    use_vllm=False,
    # vllm_mode="colocate",
    # vllm_gpu_memory_utilization=0.3,
    bf16=True,
    save_steps=50,
    logging_steps=10,
    num_train_epochs=1,
    eval_strategy="steps",
    eval_steps=50, 
    chat_template_kwargs={"enable_thinking": False},
    push_to_hub=False,  
)

trainer = GRPOTrainer(
    model=model,
    processing_class=processor,
    reward_funcs=voxel_iou_reward,
    train_dataset=train_ds_mapped,
    eval_dataset=eval_subset_mapped,
    args=training_args,
    callbacks=[FullCheckpointPushCallback(), PrintLogCallback()],
)
trainer.train(resume_from_checkpoint=f"RL_checkpoint/{checkpoint}/")
# trainer.train(resume_from_checkpoint="/rl/checkpoint-160")